# Full Combo + Conv1D, Seed-Ensembled, on Kaggle GPU

Clones `approach/full-combo-conv1d-ensemble`, forked from `approach/full-
combo-conv1d` (validated 57.6% at 100 epochs -- the current best). Same
architecture, same candidate-filtering + n-gram + vowel-guard blend,
same train/val split -- the only change is training the BiLSTM+conv
model 2-3 times with different random seeds (weight init + training-time
stochasticity only, never the train/val split, so every member stays
honestly comparable) and averaging their softmax outputs before summing
over blanks. Plain bagging: each run makes slightly different mistakes
on ambiguous states, and averaging smooths those out. Low risk since
it's the exact architecture already proven, not a new one.

**Before running:** in the notebook's Settings panel (right sidebar), set
**Accelerator = GPU T4 x2** (or any GPU) and **Internet = On** (needed to `git clone`).

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected -- check Settings > Accelerator in the sidebar')

In [ ]:
REPO_URL = "https://github.com/Sahoo-Achyutananda/MELTWATER---HACKATHON.git"
BRANCH = "approach/full-combo-conv1d-ensemble"

!rm -rf repo
!git clone --branch $BRANCH --single-branch $REPO_URL repo
%cd repo/brand-buzzword-hackathon
!ls

## Use the official competition dataset

The cloned repo carries its own copy of train.txt/test.txt (downloaded from
this same competition earlier), but overwrite them here so this notebook
verifiably sources data straight from Kaggle's own `/kaggle/input/`, not an
external GitHub copy -- same content, no ambiguity for anyone reviewing it.

In [ ]:
import shutil
shutil.copy("/kaggle/input/competitions/brand-buzzword-hackathon/train.txt", "train.txt")
shutil.copy("/kaggle/input/competitions/brand-buzzword-hackathon/test.txt", "test.txt")
print("train.txt and test.txt overwritten with the official competition dataset from /kaggle/input/")

## Train each ensemble member

Trains one BiLSTM+conv checkpoint per seed in `SEEDS` below, sequentially
in this same session -- `train_bilstm.py --seed N` always uses the exact
same train/val split (fixed regardless of seed) and only varies weight
init + training-time randomness, so every member is trained and
validated on identical word sets.

**Watch the `time=` print after epoch 1** of the first seed and multiply
by 100 epochs, then by `len(SEEDS)`, to estimate total time before this
cell finishes -- trim `SEEDS` down to fewer members (or lower `--epochs`
below) if it's tracking longer than you have runway for. Two seeds is a
reasonable default; a third is a bonus if time allows.

In [ ]:
SEEDS = [1, 2]  # trim to [1] for a single quick run, or add a 3rd if time allows
EPOCHS = 100

for seed in SEEDS:
    print(f"=== training seed {seed} ===")
    !python src/train_bilstm.py --epochs {EPOCHS} --seed {seed}

## Save checkpoints now, before anything else

Copy every trained seed checkpoint to `/kaggle/working/` immediately --
**do not wait until the end of the notebook.** If this is a "Save & Run
All" batch commit and something later (validation, submission
generation) runs out of GPU quota or time, this step already ran and the
checkpoints are safe in Output regardless of what happens after. Training
is the expensive, hard-to-repeat part; re-running validation or
submission generation later from saved checkpoints is comparatively
cheap.

In [ ]:
import glob, shutil
saved = []
for p in glob.glob("src/bilstm_conv_attn_feat_masker_seed*.pt"):
    dst = f"/kaggle/working/{p.split('/')[-1]}"
    shutil.copy(p, dst)
    saved.append(dst)
print("saved checkpoints:", saved)
assert saved, "no seed checkpoints found -- training must not have completed

## Validate the ensemble

Same held-out-train.txt methodology as every other branch -- `validate_
combined.py` auto-discovers every `bilstm_conv_attn_feat_masker_seed*.pt`
file this session just produced and averages across all of them. Compare
directly against `full-combo-conv1d`'s single-seed 57.6% -- this is the
number that decides whether the ensemble is worth submitting. Add
`--full` for the definitive number on all held-out words once you've
confirmed it's tracking above 57.6% on the quick sample.

In [ ]:
!python src/validate_combined.py

## Generate submission.csv

Only run this once the validation number is in and beats 57.6% -- it
takes a while (~250,000 words, one game at a time, now with `len(SEEDS)`
forward passes per guess instead of one) and there's no reason to spend
that time if the ensemble didn't actually help.

In [ ]:
!python src/generate_submission_combined.py

## Save outputs

Anything under `/kaggle/working/` is downloadable from the notebook's
Output tab after the run finishes.

In [ ]:
import glob, shutil
for p in glob.glob("src/bilstm_conv_attn_feat_masker_seed*.pt"):
    shutil.copy(p, f"/kaggle/working/{p.split('/')[-1]}")
shutil.copy("submission.csv", "/kaggle/working/submission.csv")
print("saved every seed checkpoint and submission.csv to /kaggle/working/ -- download from the Output tab")